# 2025 Season Retrospective
Evaluate how the model predicted the 2025 season using 2024 features → 2025 actuals (test set).

In [1]:
import sys
import os
from pathlib import Path

# Navigate to project root regardless of where kernel starts
if not Path('config.yaml').exists() and Path('../config.yaml').exists():
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.data_builder import load_config
from src.predictor import load_model_and_meta, predict_test_set, generate_enhanced_evaluation, assign_tiers
from src.feature_builder import build_shifted_dataset
from src.model_trainer import get_feature_cols, time_split, _SP_DROP_FEATURES, _RP_DROP_FEATURES

cfg = load_config()
print('Test year (feature year):', cfg['seasons']['test'])
print('\u2192 Predicting targets for year:', cfg['seasons']['test'][0] + 1)

Test year (feature year): [2024]
→ Predicting targets for year: 2025


In [2]:
# --- Batter 2025 Evaluation ---
bat_features = pd.read_csv('data/processed/batter_features.csv')
bat_shifted = build_shifted_dataset(bat_features, cfg=cfg)
_, _, bat_test = time_split(bat_shifted, cfg)

bat_model, bat_meta = load_model_and_meta('batter', cfg)
bat_test_results = predict_test_set(bat_model, bat_meta, bat_test)

# Assign rank-based tiers for tier calibration plot
bat_test_results = bat_test_results.sort_values('predicted_pts', ascending=False).reset_index(drop=True)
bat_test_results['rank'] = range(1, len(bat_test_results) + 1)
bat_test_results['tier'] = pd.cut(
    bat_test_results['rank'],
    bins=[0, 30, 75, 150, 9999],
    labels=[1, 2, 3, 4]
).astype(int)

bat_metrics = generate_enhanced_evaluation(bat_test_results, 'batter', cfg, year_label='2025')


BATTER — 2025 SEASON RETROSPECTIVE
  Players evaluated: 265
  MAE: 69.4 | RMSE: 87.1 | Spearman: 0.624 | R²: 0.424
  Bias (actual-pred): -3.6 pts (+ = we underpredict)
  Within ±50 pts: 44.5% | Within ±100 pts: 76.2%
  Top-25 precision: 48.0% | Top-50 precision: 58.0%

  TOP 10 CLOSEST PREDICTIONS:
    last  first primary_pos  predicted_pts  target_ESPN_Pts  residual
Nootbaar   Lars          RF          249.7            249.0      -0.7
Burleson   Alec          DH          317.9            316.0      -1.9
   Acuna Ronald          RF          267.0            269.0       2.0
   Marte  Ketel          2B          389.6            392.0       2.4
Gonzalez   Romy          2B          197.6            195.0      -2.6
Realmuto  J. T.           C          213.1            216.0       2.9
    Sosa  Lenyn          3B          246.6            250.0       3.4
    Happ    Ian          LF          342.2            347.0       4.8
   Wells Austin           C          208.9            214.0       5.1

C:\Users\Jake\Documents\Python\Baseball\fantasy_baseball_tool\src\predictor.py:130: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pos_mae = df.groupby("primary_pos").apply(


In [3]:
# --- Pitcher 2025 Evaluation (SP + RP) ---
pit_features = pd.read_csv('data/processed/pitcher_features.csv')
pit_shifted = build_shifted_dataset(pit_features, cfg=cfg)
_, _, pit_test = time_split(pit_shifted, cfg)

all_parts = []
for role, mask_val, drop_feats, model_key in [
    ('SP', 1, _SP_DROP_FEATURES, 'pitcher_sp_model'),
    ('RP', 0, _RP_DROP_FEATURES, 'pitcher_rp_model'),
]:
    model, meta = load_model_and_meta(model_key, cfg)
    role_test = pit_test[pit_test['is_starter'] == mask_val].copy()
    role_feature_cols = [c for c in meta['feature_cols'] if c not in drop_feats]
    role_meta = dict(meta)
    role_meta['feature_cols'] = role_feature_cols
    results = predict_test_set(model, role_meta, role_test)
    results['role'] = role
    all_parts.append(results)

pit_test_results = pd.concat(all_parts).reset_index(drop=True)

# Assign rank-based tiers for tier calibration plot
pit_test_results = pit_test_results.sort_values('predicted_pts', ascending=False).reset_index(drop=True)
pit_test_results['rank'] = range(1, len(pit_test_results) + 1)
pit_test_results['tier'] = pd.cut(
    pit_test_results['rank'],
    bins=[0, 25, 60, 120, 9999],
    labels=[1, 2, 3, 4]
).astype(int)

pit_metrics = generate_enhanced_evaluation(pit_test_results, 'pitcher', cfg, year_label='2025')


PITCHER — 2025 SEASON RETROSPECTIVE
  Players evaluated: 217
  MAE: 70.4 | RMSE: 91.8 | Spearman: 0.514 | R²: 0.330
  Bias (actual-pred): -4.4 pts (+ = we underpredict)
  Within ±50 pts: 49.8% | Within ±100 pts: 72.8%
  Top-25 precision: 44.0% | Top-50 precision: 56.0%

  TOP 10 CLOSEST PREDICTIONS:
     last   first primary_pos  predicted_pts  target_ESPN_Pts  residual
 Brazoban Huascar           P          133.7            134.0       0.3
 Ferguson   Tyler           P          103.5            105.0       1.5
  Mikolas   Miles           P          191.3            189.0      -2.3
    Zerpa   Angel           P          112.4            109.0      -3.4
  Montero  Keider           P          138.2            134.0      -4.2
Rodriguez  Yariel           P          158.3            163.0       4.7
    Perez  Martin           P          100.7             96.0      -4.7
   Vodnik  Victor           P          152.9            148.0      -4.9
    Junis   Jakob           P          140.1      

C:\Users\Jake\Documents\Python\Baseball\fantasy_baseball_tool\src\predictor.py:130: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pos_mae = df.groupby("primary_pos").apply(


In [4]:
# --- Summary Table ---
summary = pd.DataFrame([
    {'type': 'Batter', **bat_metrics},
    {'type': 'Pitcher', **pit_metrics},
]).set_index('type').round(3)
print(summary.to_string())

            mae    rmse  spearman     r2   bias  within_50  within_100  top25_precision  top50_precision
type                                                                                                    
Batter   69.379  87.110     0.624  0.424 -3.580      0.445       0.762             0.48             0.58
Pitcher  70.352  91.828     0.514  0.330 -4.406      0.498       0.728             0.44             0.56


In [5]:
# --- 2026 Draft Rankings Preview ---
bat_rank = pd.read_csv('outputs/draft_rankings_batters.csv')
pit_rank = pd.read_csv('outputs/draft_rankings_pitchers.csv')

print(f'2026 Batter Rankings: {len(bat_rank)} players')
bat_cols = ['overall_rank', 'last', 'first', 'primary_pos', 'projected_pts_2026', 'pts_2025', 'PAR', 'tier', 'risk_flags']
print(bat_rank[bat_cols].head(20).to_string(index=False))

print(f'\n2026 Pitcher Rankings: {len(pit_rank)} players')
pit_cols = ['overall_rank', 'last', 'first', 'projected_pts_2026', 'pts_2025', 'PAR', 'tier', 'risk_flags']
print(pit_rank[pit_cols].head(20).to_string(index=False))

2026 Batter Rankings: 348 players
 overall_rank        last     first primary_pos  projected_pts_2026  pts_2025        PAR  tier                 risk_flags
            1        Soto      Juan          RF          551.375816     556.0 352.284833     1                        NaN
            2     Ramirez      Jose          3B          499.050732     522.0 317.211454     1                   age_risk
            3       Judge     Aaron          RF          447.339357     599.0 248.248374     1           BABIP_regression
            4      Ohtani    Shohei          DH          436.947013     570.0 236.351467     1                        NaN
            5    Caminero    Junior          3B          417.198327     448.0 235.359048     1                        NaN
            6    Guerrero  Vladimir          1B          418.710930     448.0 232.370661     1                        NaN
            7        Witt     Bobby          SS          450.505686     461.0 227.087683     1                  

In [6]:
# --- Cell 6: FanGraphs Projection Draft Dashboard ---
import unicodedata
import re

def norm_name(s):
    return ''.join(c for c in unicodedata.normalize('NFD', str(s))
                   if unicodedata.category(c) != 'Mn').lower().strip()

def parse_pipe_md(filepath):
    """Parse pipe-delimited markdown table (Steamer format)."""
    with open(filepath, encoding='utf-8') as f:
        lines = [l for l in f.read().split('\n')
                 if '|' in l and not set(l.replace('|','').replace('-','').replace(' ','')).issubset({'-',''})]
    header = [c.strip() for c in lines[0].split('|')[1:-1]]
    rows = []
    for line in lines[1:]:
        cells = [c.strip() for c in line.split('|')[1:-1]]
        if len(cells) == len(header):
            rows.append(dict(zip(header, cells)))
    df = pd.DataFrame(rows)
    for col in df.columns:
        if col not in ('Name', 'Team'):
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def parse_tsv_md(filepath):
    """Parse tab-separated FanGraphs export (ZiPS/ZiPS-DC format)."""
    with open(filepath, encoding='utf-8') as f:
        lines = [l for l in f.read().split('\n') if '\t' in l]
    header = [c.strip() for c in lines[0].split('\t')]
    rows = [dict(zip(header, [c.strip() for c in l.split('\t')])) for l in lines[1:] if l.strip()]
    df = pd.DataFrame(rows)
    for col in df.columns:
        if col not in ('Name', 'Team'):
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def parse_article_hitters(filepath):
    """Parse plain-text hitter ranked list: '1.  Name -- Team, Pos'"""
    with open(filepath, encoding='utf-8') as f:
        lines = f.read().split('\n')
    rows = []
    for line in lines:
        m = re.match(r'^(\d+)\.\s+(.+?)\s+--', line)
        if m:
            rows.append({'article_rank': int(m.group(1)), 'Name': m.group(2).strip()})
    df = pd.DataFrame(rows)
    if not df.empty:
        df['name_key'] = df['Name'].apply(norm_name)
    return df

def parse_article_pitchers(filepath):
    """Parse plain-text pitcher ranked list: '1.  Name (SP)' or '1.  Name (RP)'"""
    with open(filepath, encoding='utf-8') as f:
        lines = f.read().split('\n')
    rows = []
    for line in lines:
        m = re.match(r'^(\d+)\.\s+(.+?)\s*\((SP|RP)\)', line)
        if m:
            rows.append({'article_rank': int(m.group(1)), 'Name': m.group(2).strip()})
    df = pd.DataFrame(rows)
    if not df.empty:
        df['name_key'] = df['Name'].apply(norm_name)
    return df

# Load all files
stm_bat = parse_pipe_md('data/fg_predictions/steamer/top_50_steamer_batters.md')
stm_pit = parse_pipe_md('data/fg_predictions/steamer/top_50_steamer_pitchers.md')
zps_bat = parse_tsv_md('data/fg_predictions/zips/top_50_zips_batters.md')
zps_pit = parse_tsv_md('data/fg_predictions/zips/top_50_zips_pitchers.md')
zdc_bat = parse_tsv_md('data/fg_predictions/zips_dc/top_50_zips_dc_batters.md')
zdc_pit = parse_tsv_md('data/fg_predictions/zips_dc/top_50_zips_dc_pitchers.md')
art_bat = parse_article_hitters('data/fg_predictions/articles/top_50_hitters_FG.md')
art_pit = parse_article_pitchers('data/fg_predictions/articles/top_50_pitchers_FG.md')

for df in [stm_bat, stm_pit, zps_bat, zps_pit, zdc_bat, zdc_pit]:
    df['name_key'] = df['Name'].apply(norm_name)

print('Parse check (rows):',
      f'stm_bat={len(stm_bat)}, zps_bat={len(zps_bat)}, zdc_bat={len(zdc_bat)}',
      f'stm_pit={len(stm_pit)}, zps_pit={len(zps_pit)}, zdc_pit={len(zdc_pit)}',
      f'art_bat={len(art_bat)}, art_pit={len(art_pit)}')

# --- Extract per-system rank ---
stm_bat['stm_rank'] = stm_bat['#'] if '#' in stm_bat.columns else stm_bat['Rank']
stm_pit['stm_rank'] = stm_pit['Rank'] if 'Rank' in stm_pit.columns else stm_pit['#']
zps_bat['zps_rank'] = zps_bat['#']
zps_pit['zps_rank'] = zps_pit['#']
zdc_bat['zdc_rank'] = zdc_bat['#']
zdc_pit['zdc_rank'] = zdc_pit['#']

# --- Build consensus batter FG table ---
BAT_STAT_COLS = ['HR', 'R', 'RBI', 'SB', 'AVG', 'wRC+', 'ADP']
_BAT_RENAME = {c: c.lower().replace('/','').replace('+','plus') for c in BAT_STAT_COLS}

bat_fg = stm_bat[['name_key'] + BAT_STAT_COLS].rename(
    columns={c: f'stm_{_BAT_RENAME[c]}' for c in BAT_STAT_COLS})

for prefix, df in [('zps', zps_bat), ('zdc', zdc_bat)]:
    renamed = df[['name_key'] + BAT_STAT_COLS].rename(
        columns={c: f'{prefix}_{_BAT_RENAME[c]}' for c in BAT_STAT_COLS})
    bat_fg = bat_fg.merge(renamed, on='name_key', how='outer')

for c in ['hr', 'r', 'rbi', 'sb', 'avg', 'wrcplus', 'adp']:
    bat_fg[f'fg_{c}'] = bat_fg[[f'stm_{c}', f'zps_{c}', f'zdc_{c}']].mean(axis=1)

bat_fg = bat_fg.merge(art_bat[['name_key', 'article_rank']], on='name_key', how='left')
bat_fg = bat_fg.merge(stm_bat[['name_key', 'stm_rank']], on='name_key', how='left')
bat_fg = bat_fg.merge(zps_bat[['name_key', 'zps_rank']], on='name_key', how='left')
bat_fg = bat_fg.merge(zdc_bat[['name_key', 'zdc_rank']], on='name_key', how='left')

# --- Build consensus pitcher FG table ---
PIT_STAT_COLS = ['IP', 'SO', 'K/9', 'ERA', 'WHIP', 'ADP']
_PIT_RENAME = {c: c.lower().replace('/','').replace('+','plus') for c in PIT_STAT_COLS}

pit_fg = stm_pit[['name_key'] + PIT_STAT_COLS].rename(
    columns={c: f'stm_{_PIT_RENAME[c]}' for c in PIT_STAT_COLS})

for prefix, df in [('zps', zps_pit), ('zdc', zdc_pit)]:
    renamed = df[['name_key'] + PIT_STAT_COLS].rename(
        columns={c: f'{prefix}_{_PIT_RENAME[c]}' for c in PIT_STAT_COLS})
    pit_fg = pit_fg.merge(renamed, on='name_key', how='outer')

for c in ['ip', 'so', 'k9', 'era', 'whip', 'adp']:
    pit_fg[f'fg_{c}'] = pit_fg[[f'stm_{c}', f'zps_{c}', f'zdc_{c}']].mean(axis=1)

pit_fg = pit_fg.merge(art_pit[['name_key', 'article_rank']], on='name_key', how='left')
pit_fg = pit_fg.merge(stm_pit[['name_key', 'stm_rank']], on='name_key', how='left')
pit_fg = pit_fg.merge(zps_pit[['name_key', 'zps_rank']], on='name_key', how='left')
pit_fg = pit_fg.merge(zdc_pit[['name_key', 'zdc_rank']], on='name_key', how='left')

print(f'\nConsensus tables: bat_fg={len(bat_fg)} rows, pit_fg={len(pit_fg)} rows')

# --- Join to draft rankings ---
bat_rank = pd.read_csv('outputs/draft_rankings_batters.csv')
pit_rank = pd.read_csv('outputs/draft_rankings_pitchers.csv')

bat_rank['name_key'] = (bat_rank['first'] + ' ' + bat_rank['last']).apply(norm_name)
pit_rank['name_key'] = (pit_rank['first'] + ' ' + pit_rank['last']).apply(norm_name)

bat_dash = bat_rank[['overall_rank', 'last', 'first', 'primary_pos',
                      'projected_pts_2026', 'pts_2025', 'PAR', 'tier', 'risk_flags', 'name_key']]\
    .merge(bat_fg[['name_key', 'fg_hr', 'fg_r', 'fg_rbi', 'fg_sb', 'fg_avg', 'fg_wrcplus', 'fg_adp',
                    'article_rank', 'stm_rank', 'zps_rank', 'zdc_rank']],
           on='name_key', how='left')

pit_dash = pit_rank[['overall_rank', 'last', 'first',
                      'projected_pts_2026', 'pts_2025', 'PAR', 'tier', 'risk_flags', 'name_key']]\
    .merge(pit_fg[['name_key', 'fg_ip', 'fg_so', 'fg_k9', 'fg_era', 'fg_whip', 'fg_adp',
                    'article_rank', 'stm_rank', 'zps_rank', 'zdc_rank']],
           on='name_key', how='left')

# rank_diff: positive = we like them more than market (ADP rank > our rank → value)
bat_dash['adp_rank'] = bat_dash['fg_adp'].round(0)
bat_dash['rank_diff'] = bat_dash['adp_rank'] - bat_dash['overall_rank']
bat_dash['signal'] = bat_dash['rank_diff'].apply(
    lambda x: 'VALUE' if x > 20 else ('AVOID' if x < -20 else ''))

pit_dash['adp_rank'] = pit_dash['fg_adp'].round(0)
pit_dash['rank_diff'] = pit_dash['adp_rank'] - pit_dash['overall_rank']
pit_dash['signal'] = pit_dash['rank_diff'].apply(
    lambda x: 'VALUE' if x > 20 else ('AVOID' if x < -20 else ''))

# Match rate check
bat_matched = bat_dash.head(50)['fg_adp'].notna().sum()
pit_matched = pit_dash.head(50)['fg_adp'].notna().sum()
print(f'FG match rate (top 50): batters={bat_matched}/50, pitchers={pit_matched}/50')

# --- Print batter dashboard ---
print('\n' + '='*130)
print('BATTER DRAFT DASHBOARD — Top 30 by Our Model Rank')
print('='*130)
bat_cols = ['overall_rank', 'last', 'first', 'primary_pos', 'projected_pts_2026',
            'adp_rank', 'rank_diff', 'stm_rank', 'zps_rank', 'zdc_rank', 'article_rank',
            'fg_hr', 'fg_r', 'fg_rbi', 'fg_sb', 'fg_avg', 'fg_wrcplus', 'signal']
bat_top = bat_dash.head(30)[bat_cols].copy()
bat_top['fg_avg'] = bat_top['fg_avg'].round(3)
bat_top['fg_hr'] = bat_top['fg_hr'].round(0)
bat_top['fg_r'] = bat_top['fg_r'].round(0)
bat_top['fg_rbi'] = bat_top['fg_rbi'].round(0)
bat_top['fg_sb'] = bat_top['fg_sb'].round(0)
bat_top['fg_wrcplus'] = bat_top['fg_wrcplus'].round(0)
print(bat_top.to_string(index=False))

# --- Print pitcher dashboard ---
print('\n' + '='*125)
print('PITCHER DRAFT DASHBOARD — Top 30 by Our Model Rank')
print('='*125)
pit_cols = ['overall_rank', 'last', 'first', 'projected_pts_2026',
            'adp_rank', 'rank_diff', 'stm_rank', 'zps_rank', 'zdc_rank', 'article_rank',
            'fg_ip', 'fg_so', 'fg_era', 'fg_whip', 'fg_k9', 'signal']
pit_top = pit_dash.head(30)[pit_cols].copy()
pit_top['fg_ip'] = pit_top['fg_ip'].round(1)
pit_top['fg_so'] = pit_top['fg_so'].round(0)
pit_top['fg_era'] = pit_top['fg_era'].round(2)
pit_top['fg_whip'] = pit_top['fg_whip'].round(2)
pit_top['fg_k9'] = pit_top['fg_k9'].round(2)
print(pit_top.to_string(index=False))

# --- Value / Avoid summaries ---
bat_value = bat_dash[bat_dash['signal'] == 'VALUE'].sort_values('overall_rank')
bat_avoid = bat_dash[bat_dash['signal'] == 'AVOID'].sort_values('overall_rank')
pit_value = pit_dash[pit_dash['signal'] == 'VALUE'].sort_values('overall_rank')
pit_avoid = pit_dash[pit_dash['signal'] == 'AVOID'].sort_values('overall_rank')

sum_cols_bat = ['overall_rank', 'last', 'first', 'primary_pos', 'projected_pts_2026', 'adp_rank', 'rank_diff', 'fg_hr', 'fg_r', 'fg_rbi', 'fg_sb']
sum_cols_pit = ['overall_rank', 'last', 'first', 'projected_pts_2026', 'adp_rank', 'rank_diff', 'fg_ip', 'fg_so', 'fg_era']

print(f'\n--- BATTER VALUE PICKS (model rank >> ADP) — {len(bat_value)} players ---')
print(bat_value[sum_cols_bat].to_string(index=False) if not bat_value.empty else '  (none)')

print(f'\n--- BATTER POTENTIAL FADES (ADP >> model rank) — {len(bat_avoid)} players ---')
print(bat_avoid[sum_cols_bat].to_string(index=False) if not bat_avoid.empty else '  (none)')

print(f'\n--- PITCHER VALUE PICKS (model rank >> ADP) — {len(pit_value)} players ---')
print(pit_value[sum_cols_pit].to_string(index=False) if not pit_value.empty else '  (none)')

print(f'\n--- PITCHER POTENTIAL FADES (ADP >> model rank) — {len(pit_avoid)} players ---')
print(pit_avoid[sum_cols_pit].to_string(index=False) if not pit_avoid.empty else '  (none)')

# --- Comparison Graphs: Our Model vs FG ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Scatter: Batter Our Rank vs FG ADP
ax = axes[0, 0]
matched = bat_dash.dropna(subset=['fg_adp']).head(50)
ax.scatter(matched['overall_rank'], matched['fg_adp'], alpha=0.7)
lim = max(matched['overall_rank'].max(), matched['fg_adp'].max()) + 5
ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='Agreement')
for _, r in matched.iterrows():
    if abs(r['rank_diff']) > 20:
        ax.annotate(r['last'], (r['overall_rank'], r['fg_adp']), fontsize=7)
ax.set_xlabel('Our Model Rank')
ax.set_ylabel('FG Consensus ADP')
ax.set_title('Batters: Our Rank vs FG ADP')
ax.legend()

# Scatter: Pitcher Our Rank vs FG ADP
ax = axes[0, 1]
matched_p = pit_dash.dropna(subset=['fg_adp']).head(50)
ax.scatter(matched_p['overall_rank'], matched_p['fg_adp'], alpha=0.7, color='orange')
lim_p = max(matched_p['overall_rank'].max(), matched_p['fg_adp'].max()) + 5
ax.plot([0, lim_p], [0, lim_p], 'k--', alpha=0.3, label='Agreement')
for _, r in matched_p.iterrows():
    if abs(r['rank_diff']) > 20:
        ax.annotate(r['last'], (r['overall_rank'], r['fg_adp']), fontsize=7)
ax.set_xlabel('Our Model Rank')
ax.set_ylabel('FG Consensus ADP')
ax.set_title('Pitchers: Our Rank vs FG ADP')
ax.legend()

# Bar: Batter rank_diff (top 30)
ax = axes[1, 0]
top30_bat = bat_dash.dropna(subset=['rank_diff']).head(30).sort_values('rank_diff')
colors = ['green' if x > 20 else ('red' if x < -20 else 'gray') for x in top30_bat['rank_diff']]
ax.barh(top30_bat['last'], top30_bat['rank_diff'], color=colors)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('ADP Rank − Our Rank (positive = value)')
ax.set_title('Batter Rank Difference (Top 30)')

# Bar: Pitcher rank_diff (top 30)
ax = axes[1, 1]
top30_pit = pit_dash.dropna(subset=['rank_diff']).head(30).sort_values('rank_diff')
colors_p = ['green' if x > 20 else ('red' if x < -20 else 'gray') for x in top30_pit['rank_diff']]
ax.barh(top30_pit['last'], top30_pit['rank_diff'], color=colors_p)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('ADP Rank − Our Rank (positive = value)')
ax.set_title('Pitcher Rank Difference (Top 30)')

fig.tight_layout()
fig.savefig('outputs/figures/model_evaluation/draft_dashboard_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

Parse check (rows): stm_bat=49, zps_bat=50, zdc_bat=50 stm_pit=50, zps_pit=50, zdc_pit=50 art_bat=50, art_pit=50

Consensus tables: bat_fg=64 rows, pit_fg=69 rows
FG match rate (top 50): batters=29/50, pitchers=27/50

BATTER DRAFT DASHBOARD — Top 30 by Our Model Rank
 overall_rank        last     first primary_pos  projected_pts_2026  adp_rank  rank_diff  stm_rank  zps_rank  zdc_rank  article_rank  fg_hr  fg_r  fg_rbi  fg_sb  fg_avg  fg_wrcplus signal
            1        Soto      Juan          RF          551.375816       5.0        4.0       6.0       5.0       5.0           4.0   36.0 109.0    99.0   22.0   0.272       162.0       
            2     Ramirez      Jose          3B          499.050732       6.0        4.0      17.0      12.0      15.0           5.0   27.0  90.0    90.0   29.0   0.272       124.0       
            3       Judge     Aaron          RF          447.339357       2.0       -1.0       1.0       1.0       1.0           2.0   42.0 109.0   112.0    8.0   0.286